In [12]:
import pandas as pd
import geopandas as gpd
from pyprojroot import here

In [13]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [14]:
mun = gpd.read_file(RAW_DATA_DIR/"geography/municipalities/00mun.shp")
rain = gpd.read_file(RAW_DATA_DIR/"resources/rainfall_shp/precipitacionMediaAnual.shp")

In [15]:
# We verify that both gpd share the same CRS
if mun.crs != rain.crs:
    rain = rain.to_crs(mun.crs)

In [16]:
# CLAVE is the average annual rainfall in mm
rain['CLAVE'].unique()
mapping = {'CANEVA': '0', 'H2O': '0', 'P/E': '0'}
rain['CLAVE'] = rain['CLAVE'].replace(mapping)

In [17]:
joined = gpd.sjoin(mun, rain[["CLAVE", "geometry"]], how="left", predicate="intersects")

In [18]:
result = joined[["CVEGEO", "CLAVE"]].drop_duplicates(subset="CVEGEO")
result.fillna('0', inplace=True)
result.rename(columns={'CLAVE': 'rainfall'}, inplace=True)
result['rainfall'] = result['rainfall'].astype(int)

In [19]:
result.to_csv(INTERIM_DATA_DIR/"rainfall_by_municipality.csv", index=False)

print(f"Done. {len(result)} municipalities written.")
print(result.sample(10))

Done. 2478 municipalities written.
     CVEGEO  rainfall
158   07060      3500
916   17001      1000
1196  20553       700
927   17011      1000
2271  30005      1500
1296  20077         0
2473  32004         0
541   13040      1200
337   11036         0
26    04012      1100
